[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/07_Dot_Product_Similarity_and_Patterns.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/07_Dot_Product_Similarity_and_Patterns.ipynb)

# The Dot Product — How Machine Learning Measures Similarity

The dot product is the single operation underneath almost everything in ML:

- **Linear regression**: `prediction = w · x` (weights dot features)
- **Neural networks**: every neuron computes a dot product before applying an activation
- **Transformer attention**: `score(query, key) = q · k` — how much should token A attend to token B?
- **Embeddings**: similar concepts have vectors that point in the same direction

All of these use the same geometric fact: **the dot product measures how aligned two vectors are**.

---

**Seven things this notebook makes concrete**

1. What a dot product *actually* measures — alignment, not just multiplication
2. How the angle between two vectors determines the dot product
3. Why the weight vector is the *pattern the model learned to look for*
4. Why prediction = dot product = similarity score
5. Why a zero feature value is a missing dimension — geometric silence
6. How every neuron in a neural network is a dot product with an activation
7. Why the gradient is also a dot product — the same operation that predicts also learns


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Arc, FancyArrowPatch
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

def draw_vec(ax, vec, label='', color='#1565C0', origin=None, lw=2.2, fs=11):
    if origin is None:
        origin = [0.0, 0.0]
    ox, oy = origin
    vx, vy = vec
    ax.annotate('', xy=(ox+vx, oy+vy), xytext=(ox, oy),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw, mutation_scale=22))
    if label:
        tip = np.array([ox + vx * 1.18, oy + vy * 1.18])
        ax.text(tip[0], tip[1], label, color=color, fontsize=fs,
                fontweight='bold', ha='center', va='center')

print('Ready.')


## Part 1 — What a Dot Product Actually Measures

Given two vectors **a** and **b**, the dot product is:

```
a · b  =  a₁b₁ + a₂b₂ + … + aₙbₙ       (component form)
       =  |a| × |b| × cos(θ)              (geometric form)
```

The component form looks like plain multiplication and addition.
The geometric form reveals what it *means*:

- **θ = 0°** — same direction — `cos(0) = 1` — maximum positive
- **θ = 90°** — perpendicular — `cos(90) = 0` — zero, no alignment
- **θ = 180°** — opposite directions — `cos(180) = -1` — maximum negative

The dot product is zero not because the vectors are small, but because they are **perpendicular** — they share no common direction.


In [ ]:
w = np.array([2.0, 1.0])   # fixed reference vector

cases = [
    (np.array([1.6, 0.8]),  '#4CAF50', 'Aligned',       'same direction'),
    (np.array([-0.5, 1.0]), '#FF9800', 'Perpendicular',  'no shared direction'),
    (np.array([-1.6, -0.8]),'#e57373', 'Opposite',       'pointing away'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (x, color, title, note) in zip(axes[:3], cases):
    dp    = np.dot(w, x)
    costh = dp / (np.linalg.norm(w) * np.linalg.norm(x))
    theta = np.degrees(np.arccos(np.clip(costh, -1, 1)))

    draw_vec(ax, w, label='w', color='#1565C0')
    draw_vec(ax, x, label='x', color=color)

    # angle arc
    r = 0.45
    t1 = np.degrees(np.arctan2(w[1], w[0]))
    t2 = np.degrees(np.arctan2(x[1], x[0]))
    lo, hi = min(t1, t2), max(t1, t2)
    if hi - lo > 180:
        lo, hi = hi, lo + 360
    arc = Arc((0, 0), r*2, r*2, angle=0, theta1=lo, theta2=hi, color='gray', lw=1.5)
    ax.add_patch(arc)
    mid = np.radians((lo + hi) / 2)
    ax.text(r*1.5*np.cos(mid), r*1.5*np.sin(mid), f'{theta:.0f}°',
            ha='center', va='center', fontsize=9, color='gray')

    ax.set_xlim(-2.5, 2.8)
    ax.set_ylim(-1.8, 1.8)
    ax.axhline(0, color='lightgray', lw=0.8)
    ax.axvline(0, color='lightgray', lw=0.8)
    ax.set_aspect('equal')
    ax.set_title(f'{title}   dot = {dp:.2f}', fontsize=11)
    ax.set_xlabel(note, fontsize=9)

# cos(θ) curve
theta_range = np.linspace(0, 180, 300)
cos_vals    = np.cos(np.radians(theta_range))
axes[3].plot(theta_range, cos_vals, color='#1565C0', lw=2.5)
axes[3].axhline(0, color='black', lw=0.9, linestyle='--')
for th, lab, color in [(0,'θ=0\naligned','#4CAF50'),
                       (90,'θ=90\nno alignment','#FF9800'),
                       (180,'θ=180\nopposite','#e57373')]:
    cv = np.cos(np.radians(th))
    axes[3].scatter([th], [cv], s=80, color=color, zorder=5)
    axes[3].text(th + 5, cv + 0.08, lab, fontsize=8, color=color)
axes[3].set_xlabel('Angle θ between vectors (degrees)')
axes[3].set_ylabel('cos(θ)')
axes[3].set_title('How angle controls the dot product', fontsize=11)

plt.suptitle('dot product = |w| × |x| × cos(θ)   —   alignment determines the value',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## Part 2 — The Weight Vector Is the Pattern the Model Learned

After training, the weight vector **w** is not random. It points in the direction of the pattern that best predicts the target.

Think of **w** as the model's *ideal profile* — the combination of feature values associated with the outcome.

For every new observation **x**, the prediction is:

```
prediction  =  w · x  =  |w| × |x| × cos(θ)
```

- **x aligns with w** (similar profile to the ideal) → large positive dot product → high prediction
- **x is perpendicular to w** (unrelated profile) → dot product near zero → near-average prediction
- **x opposes w** (opposite of the ideal) → negative dot product → low prediction

The model does not "check rules". It asks: **how similar is this observation's feature vector to the pattern vector w?**


In [ ]:
# 2-feature world: income (0-1) and age (0-1), both normalised
# True pattern: high income + older age → high credit score
np.random.seed(7)
n = 200
income_n = np.random.uniform(0, 1, n)
age_n    = np.random.uniform(0, 1, n)
X2 = np.column_stack([income_n, age_n])

w2 = np.array([0.7, 0.5])   # the "true" weight vector
y2 = X2 @ w2 + np.random.normal(0, 0.08, n)

dot_products = X2 @ w2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: observations coloured by dot product ────────────────────────────────
sc = axes[0].scatter(income_n, age_n, c=dot_products, cmap='RdYlGn',
                     s=30, alpha=0.8, edgecolors='none')
plt.colorbar(sc, ax=axes[0], label='w · x  (alignment score)')
draw_vec(axes[0], w2 * 0.9, label='w', color='#1565C0', fs=13)
axes[0].set_xlabel('income (normalised)')
axes[0].set_ylabel('age (normalised)')
axes[0].set_title('Observations coloured by their dot product with w', fontsize=10)
axes[0].set_xlim(-0.1, 1.15)
axes[0].set_ylim(-0.1, 1.15)

# show three labelled example points
for label, point, color in [
    ('High\nalignment', np.array([0.9, 0.85]), '#1B5E20'),
    ('Perpendicular\n(near zero)', np.array([0.4, -0.1+0.5]), '#E65100'),
    ('Low\nalignment', np.array([0.05, 0.08]), '#B71C1C'),
]:
    axes[0].scatter(*point, s=120, color=color, zorder=5, edgecolors='white', lw=1.5)
    axes[0].annotate(label, xy=point, xytext=(point[0]+0.05, point[1]-0.12),
                     fontsize=7.5, color=color,
                     arrowprops=dict(arrowstyle='->', color=color, lw=1))

# ── Right: dot product vs actual target ───────────────────────────────────────
axes[1].scatter(dot_products, y2, alpha=0.4, s=18, color='#1565C0')
m, b = np.polyfit(dot_products, y2, 1)
xs = np.linspace(dot_products.min(), dot_products.max(), 100)
axes[1].plot(xs, m*xs+b, color='#e57373', lw=2.5, label='linear fit')
axes[1].set_xlabel('w · x  (dot product / alignment score)')
axes[1].set_ylabel('Target value')
axes[1].set_title('Dot product predicts the target — more aligned = higher value', fontsize=10)
axes[1].legend(fontsize=9)

plt.suptitle('The weight vector w defines the pattern — dot product measures how well each observation fits it',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print('Observations in the top-right (high income, older age) are most aligned with w.')
print('Their dot product is largest — they score highest on the prediction.')
print('The bottom-left (low income, young) point in the opposite direction — lowest score.')


## Part 3 — Prediction = Dot Product = Similarity Score

In the 2-feature world: `prediction = w₁x₁ + w₂x₂ = w · x`

In a full model with many features: `prediction = Σⱼ wⱼ xⱼ = w · x`

The same formula, regardless of how many features. Each term `wⱼ xⱼ` is the contribution of one dimension to the total alignment.

**Why the intercept exists**

The raw dot product `w · x` is zero when x is perpendicular to w — but that does not mean the prediction should be zero. The intercept shifts the whole prediction up or down so the average prediction matches the average target.

```
prediction  =  b  +  w · x
            =  b  +  (alignment of x with w)
```

`b` sets the baseline. `w · x` moves each observation above or below that baseline based on how well its features match the learned pattern.


In [ ]:
# Show prediction as a sum of per-dimension alignments
# Use a concrete 5-feature example from the insurance domain
np.random.seed(42)
n = 400
age_r     = np.random.uniform(25, 65, n)
bmi_r     = np.random.uniform(18, 40, n)
act_r     = np.random.choice([1,2,3,4], n, p=[0.25,0.35,0.25,0.15]).astype(float)
smoker_r  = np.random.choice([0,1], n, p=[0.75,0.25]).astype(float)
chronic_r = np.random.choice([0,1], n, p=[0.70,0.30]).astype(float)

TRUE_W2       = np.array([50., 30., -200., 3500., 2500.])
TRUE_B2       = 3000.
FEATURES2     = ['age', 'bmi', 'activity_level', 'smoker', 'has_chronic']
X5 = np.column_stack([age_r, bmi_r, act_r, smoker_r, chronic_r])
y5 = TRUE_B2 + X5 @ TRUE_W2 + np.random.normal(0, 300, n)

# Pick two contrasting observations
obs_a = np.array([55., 35., 1., 1., 1.])   # old, high bmi, sedentary, smoker, chronic
obs_b = np.array([28., 20., 4., 0., 0.])   # young, low bmi, active, non-smoker, healthy

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, obs, label, bcolor in [
    (axes[0], obs_a, 'Person A  (smoker, chronic, sedentary)', '#e57373'),
    (axes[1], obs_b, 'Person B  (young, active, no risk factors)', '#4CAF50'),
]:
    contribs = TRUE_W2 * obs
    colors   = ['#4CAF50' if c > 0 else '#e57373' for c in contribs]
    bars = ax.barh(FEATURES2, contribs, color=colors, edgecolor='white', height=0.55)
    ax.axvline(0, color='black', lw=1.2)
    ax.set_xlabel('Contribution to premium  (w_j × x_j)')
    ax.set_title(label, fontsize=10)
    total = TRUE_B2 + contribs.sum()
    ax.text(0.98, 0.04, f'Total: ${total:,.0f}', transform=ax.transAxes,
            ha='right', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=bcolor, alpha=0.3))
    for bar, val in zip(bars, contribs):
        xpos = val + 50 if val >= 0 else val - 50
        ha   = 'left' if val >= 0 else 'right'
        ax.text(xpos, bar.get_y() + bar.get_height()/2,
                f'${val:+,.0f}', va='center', ha=ha, fontsize=8)

plt.suptitle('Prediction = intercept + Σ(w_j × x_j)  =  baseline + total alignment across all dimensions',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print(f'Person A: each risk feature is active → large positive dot product → premium ${TRUE_B2 + (TRUE_W2 * obs_a).sum():,.0f}')
print(f'Person B: risk features are zero → those dimensions contribute nothing → premium ${TRUE_B2 + (TRUE_W2 * obs_b).sum():,.0f}')


## Part 4 — Zero Feature Value = Missing Dimension = Geometric Silence

When a feature value is zero, its term in the dot product vanishes:

```
wⱼ × 0  =  0
```

Geometrically, the observation vector **x** has **no component in dimension j**.
It does not point toward or away from that dimension — it simply does not exist there.

Imagine a 3D space where the axes are `income`, `age`, and `smoker`.

- A non-smoker's feature vector has a zero in the `smoker` dimension
- The weight `w_smoker` pulls in the smoker direction
- Their dot product contribution from that dimension is exactly zero
- The observation is *flat* in that direction — it contributes nothing

This is why binary and categorical features are silent for observations where they are 0 — it is not a special rule, it is geometry.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Left: 2D — both dimensions active ─────────────────────────────────────────
ax = axes[0]
w3 = np.array([1.5, 2.0])
x_full = np.array([1.2, 1.6])
dp_full = np.dot(w3, x_full)

draw_vec(ax, w3, label='w', color='#1565C0')
draw_vec(ax, x_full, label='x', color='#4CAF50')

# projection of x onto w
proj_len = np.dot(x_full, w3) / np.linalg.norm(w3)
proj_vec = proj_len * w3 / np.linalg.norm(w3)
draw_vec(ax, proj_vec, color='#FF9800', lw=1.5)
ax.plot([x_full[0], proj_vec[0]], [x_full[1], proj_vec[1]],
        '--', color='gray', lw=1.2)
ax.text(proj_vec[0]/2 - 0.3, proj_vec[1]/2 + 0.1, 'projection', fontsize=8, color='#FF9800')

ax.set_xlim(-0.3, 2.8)
ax.set_ylim(-0.3, 2.8)
ax.set_aspect('equal')
ax.axhline(0, color='lightgray', lw=0.8)
ax.axvline(0, color='lightgray', lw=0.8)
ax.set_xlabel('feature 1  (income)')
ax.set_ylabel('feature 2  (age)')
ax.set_title(f'Both features active:  w·x = {dp_full:.2f}', fontsize=10)

# ── Middle: feature 1 = 0 (missing dimension) ─────────────────────────────────
ax = axes[1]
x_miss1 = np.array([0.0, 1.6])   # income=0, age=1.6
dp_miss1 = np.dot(w3, x_miss1)

draw_vec(ax, w3, label='w', color='#1565C0')
draw_vec(ax, x_miss1, label='x', color='#e57373')

proj_len2 = np.dot(x_miss1, w3) / np.linalg.norm(w3)
proj_vec2 = proj_len2 * w3 / np.linalg.norm(w3)
draw_vec(ax, proj_vec2, color='#FF9800', lw=1.5)
ax.plot([x_miss1[0], proj_vec2[0]], [x_miss1[1], proj_vec2[1]],
        '--', color='gray', lw=1.2)

ax.axvline(0, color='#e57373', lw=1.5, linestyle=':', label='stuck on axis (x₁=0)')
ax.set_xlim(-0.3, 2.8)
ax.set_ylim(-0.3, 2.8)
ax.set_aspect('equal')
ax.axhline(0, color='lightgray', lw=0.8)
ax.set_xlabel('feature 1  (income = 0)')
ax.set_ylabel('feature 2  (age)')
ax.set_title(f'Income=0: only age contributes  w·x = {dp_miss1:.2f}', fontsize=10)
ax.legend(fontsize=8)

# ── Right: both features = 0 ──────────────────────────────────────────────────
ax = axes[2]
x_zero = np.array([0.0, 0.0])

draw_vec(ax, w3, label='w', color='#1565C0')
ax.scatter([0], [0], s=120, color='#e57373', zorder=5, label='x = (0, 0)')
ax.text(0.15, 0.05, 'x is at the origin\n— no direction at all', fontsize=9,
        color='#e57373')
ax.set_xlim(-0.3, 2.8)
ax.set_ylim(-0.3, 2.8)
ax.set_aspect('equal')
ax.axhline(0, color='lightgray', lw=0.8)
ax.axvline(0, color='lightgray', lw=0.8)
ax.set_xlabel('feature 1')
ax.set_ylabel('feature 2')
ax.set_title(f'Both features=0:  w·x = 0  (complete silence)', fontsize=10)
ax.legend(fontsize=8)

plt.suptitle('Zero feature value = the observation has no component in that dimension = contributes 0 to dot product',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print('Left:   x has components in both dimensions — both contribute to w·x')
print('Middle: x has no income component (x₁=0) — only age contributes — smaller dot product')
print('Right:  x is at the origin — no direction — dot product is zero regardless of w')


## Part 5 — Each Neuron in a Neural Network Is a Dot Product

A single neuron in a neural network does exactly two things:

```
z  =  w · x  +  b          (dot product — linear combination of inputs)
output  =  activation(z)    (non-linearity — decides whether the neuron fires)
```

The weight vector **w** of each neuron defines the pattern that neuron is looking for.
The dot product measures how strongly the input matches that pattern.
The activation function decides whether the match is strong enough to pass on.

**Why multiple neurons?** A single dot product can only detect one pattern. A layer with k neurons computes k different dot products — each neuron specializes in detecting a different aspect of the input.

```
layer output  =  activation( X @ W + b )
```

where `X @ W` is a matrix of dot products — every input dotted with every neuron's weight vector simultaneously.

**Why non-linearity matters**: stacking pure dot products (linear layers) collapses to a single dot product. The activation function breaks this, allowing the network to detect combinations of patterns — not just single patterns.


In [ ]:
# 2D input space — three neurons with different weight vectors
# Each neuron fires strongly for a different region of the input space

np.random.seed(0)
n_pts  = 400
X_grid = np.random.uniform(-2, 2, (n_pts, 2))

# Three neurons: each detects a different direction in 2D
neuron_w = np.array([
    [ 1.0,  0.5],   # neuron 0: mostly feature-1
    [-0.5,  1.0],   # neuron 1: mostly feature-2
    [ 0.8, -0.8],   # neuron 2: feature-1 minus feature-2
])
neuron_colors = ['#1565C0', '#4CAF50', '#E65100']
neuron_labels = ['Neuron 0', 'Neuron 1', 'Neuron 2']

def relu(z):
    return np.maximum(0, z)

fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))

for i, (w_n, color, label) in enumerate(zip(neuron_w, neuron_colors, neuron_labels)):
    z      = X_grid @ w_n           # dot product (pre-activation)
    output = relu(z)                 # ReLU activation

    sc = axes[i].scatter(X_grid[:,0], X_grid[:,1], c=output,
                         cmap='Blues', s=18, alpha=0.85, edgecolors='none',
                         vmin=0, vmax=output.max())
    plt.colorbar(sc, ax=axes[i], label='ReLU output')
    draw_vec(axes[i], w_n * 0.8, label='w', color=color, fs=10)
    axes[i].set_xlim(-2.4, 2.8)
    axes[i].set_ylim(-2.4, 2.4)
    axes[i].set_aspect('equal')
    axes[i].axhline(0, color='gray', lw=0.7, linestyle='--')
    axes[i].axvline(0, color='gray', lw=0.7, linestyle='--')
    axes[i].set_title(f'{label}   fires for points\naligned with w', fontsize=9)
    axes[i].set_xlabel('feature 1')

# 4th panel: combine all three — show which neuron fires most strongly
all_z      = X_grid @ neuron_w.T         # (n, 3)
winner     = np.argmax(relu(all_z), axis=1)
win_colors = np.array(neuron_colors)[winner]
axes[3].scatter(X_grid[:,0], X_grid[:,1], c=win_colors, s=18, alpha=0.85, edgecolors='none')
for w_n, color, label in zip(neuron_w, neuron_colors, neuron_labels):
    draw_vec(axes[3], w_n * 0.8, color=color, fs=9)
    axes[3].scatter([], [], color=color, label=label, s=40)
axes[3].legend(fontsize=8, loc='lower right')
axes[3].set_xlim(-2.4, 2.8)
axes[3].set_ylim(-2.4, 2.4)
axes[3].set_aspect('equal')
axes[3].axhline(0, color='gray', lw=0.7, linestyle='--')
axes[3].axvline(0, color='gray', lw=0.7, linestyle='--')
axes[3].set_title('Which neuron fires most strongly?\n(max ReLU output wins)', fontsize=9)
axes[3].set_xlabel('feature 1')

plt.suptitle('Each neuron = one dot product + one activation  —  different w vectors detect different patterns',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print('Each neuron fires (ReLU > 0) only for points in its preferred direction.')
print('A neural network layer runs ALL neurons in parallel — one dot product each.')
print('The combination of many pattern detectors allows the network to carve up input space.')


## Part 6 — Attention: Dot Product as Token Similarity

In Transformer models, attention asks: *for each token, which other tokens are most relevant?*

The answer is computed with dot products:

```
score(query, key)  =  q · k  /  √d
attention_weight   =  softmax( scores )
output             =  Σ attention_weight × value
```

- **Query** (q): what this token is looking for
- **Key** (k): what each other token offers
- **Value** (v): what each token actually sends if attended to

`q · k` measures alignment between what one token seeks and what another provides.
Large dot product → high attention weight → that token is attended to more.

The `√d` scaling prevents the dot products from becoming too large when the vector dimension d is high (large d → large magnitudes → extreme softmax values → vanishing gradients).

This is exactly the same geometric intuition: **similar tokens point in similar directions, giving a large dot product**.


In [ ]:
# Small 2D token embeddings — manually placed to show semantic grouping
tokens = ['bank', 'river', 'money', 'flow', 'loan', 'stream']

# 2D embeddings: finance cluster and water cluster
embeddings = np.array([
    [ 0.9,  0.2],   # bank      — between finance and water
    [-0.8,  0.7],   # river     — water
    [ 1.0, -0.3],   # money     — finance
    [-0.6,  0.9],   # flow      — water
    [ 0.8, -0.5],   # loan      — finance
    [-0.9,  0.5],   # stream    — water
])

# Normalise to unit vectors for clean cosine similarity
norms  = np.linalg.norm(embeddings, axis=1, keepdims=True)
emb_n  = embeddings / norms

# Dot product similarity matrix (= cosine similarity since vectors are unit)
sim_matrix = emb_n @ emb_n.T

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: 2D embedding space ──────────────────────────────────────────────────
colors_tok = ['#1565C0','#26A69A','#1565C0','#26A69A','#1565C0','#26A69A']
axes[0].scatter(embeddings[:,0], embeddings[:,1],
                c=colors_tok, s=150, zorder=5, edgecolors='white', lw=1.5)
for i, tok in enumerate(tokens):
    axes[0].text(embeddings[i,0]+0.05, embeddings[i,1]+0.08, tok,
                 fontsize=10, fontweight='bold',
                 color='#1565C0' if colors_tok[i]=='#1565C0' else '#26A69A')
axes[0].axhline(0, color='lightgray', lw=0.8)
axes[0].axvline(0, color='lightgray', lw=0.8)
axes[0].set_xlabel('embedding dim 1')
axes[0].set_ylabel('embedding dim 2')
axes[0].set_title('2D token embedding space', fontsize=11)
axes[0].scatter([],[], color='#1565C0', label='finance cluster', s=60)
axes[0].scatter([],[], color='#26A69A', label='water cluster', s=60)
axes[0].legend(fontsize=9)

# draw lines between tokens with high similarity (>0.7)
for i in range(len(tokens)):
    for j in range(i+1, len(tokens)):
        if sim_matrix[i,j] > 0.7:
            axes[0].plot([embeddings[i,0], embeddings[j,0]],
                         [embeddings[i,1], embeddings[j,1]],
                         color='gray', lw=1.0, alpha=0.5, linestyle='--')

# ── Right: attention heatmap using "bank" as query ────────────────────────────
query_idx = 0   # "bank"
scores    = sim_matrix[query_idx]
softmax_s = np.exp(scores) / np.exp(scores).sum()

bars = axes[1].barh(tokens[::-1], softmax_s[::-1],
                    color=['#e57373' if t == tokens[query_idx] else '#4CAF50'
                           for t in tokens[::-1]])
axes[1].set_xlabel('Attention weight  (softmax of q · k)')
axes[1].set_title(f'Attention from query token: "{tokens[query_idx]}"', fontsize=11)
for bar, val in zip(bars, softmax_s[::-1]):
    axes[1].text(val + 0.003, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Transformer attention: q · k measures how relevant each token is to the query',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print(f'Query: "bank"')
print(f'Highest attention → tokens in the same direction in embedding space')
print()
for tok, score, weight in sorted(zip(tokens, sim_matrix[0], softmax_s),
                                  key=lambda x: -x[2]):
    print(f'  {tok:<10}  cos similarity = {score:>6.3f}   attention = {weight:.3f}')


## Part 7 — The Gradient Is a Dot Product Too

From Notebook 06, the gradient for weight j is:

```
gradient_j  =  -(2/n) × Σᵢ( errorᵢ × xᵢⱼ )
             =  -(2/n) × (errors · column_j_of_X)
```

`errors · column_j` is a dot product between two n-dimensional vectors:
- The **error vector** (one entry per observation: how wrong was the model for that person?)
- **Feature column j** (one entry per observation: what was that person's value for feature j?)

**What this dot product measures**: how correlated are the model's errors with feature j?

- Large positive dot product → errors are aligned with feature j → model consistently under-predicts when feature j is large → weight j needs to increase
- Near zero → errors have no pattern along feature j → weight j is already well calibrated
- Large negative → model over-predicts when feature j is large → weight j needs to decrease

Training is the process of driving every `errors · column_j` toward zero — the error vector becomes perpendicular to every feature column — no more learnable signal remains.


In [ ]:
from sklearn.preprocessing import StandardScaler as _SS

# Use the 5-feature insurance dataset from above, with partially trained weights
# Run just 50 epochs so there is still meaningful gradient signal
X5_sc   = _SS().fit_transform(X5)
n5      = len(y5)
w5, b5  = np.zeros(5), 0.0

for _ in range(50):
    yp    = X5_sc @ w5 + b5
    err   = y5 - yp
    w5   -= 0.05 * (-(2/n5) * (X5_sc.T @ err))
    b5   -= 0.05 * (-(2/n5) * err.sum())

# Error vector at epoch 50
yp50      = X5_sc @ w5 + b5
errors50  = y5 - yp50

# Dot product of errors with each feature column = gradient signal
dot_err_feat = errors50 @ X5_sc    # shape (5,)
gradient     = -(2/n5) * dot_err_feat

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: dot product of errors with each feature column ──────────────────────
colors_dp = ['#4CAF50' if v > 0 else '#e57373' for v in dot_err_feat]
bars = axes[0].barh(FEATURES2, dot_err_feat, color=colors_dp, edgecolor='white', height=0.55)
axes[0].axvline(0, color='black', lw=1.2)
axes[0].set_xlabel('errors · feature_column_j   (dot product)')
axes[0].set_title('How aligned are errors with each feature?  (epoch 50 — not yet converged)', fontsize=10)
for bar, val in zip(bars, dot_err_feat):
    xpos = val + abs(dot_err_feat).max()*0.01 if val >= 0 else val - abs(dot_err_feat).max()*0.01
    ha   = 'left' if val >= 0 else 'right'
    axes[0].text(xpos, bar.get_y()+bar.get_height()/2,
                 f'{val:,.0f}', va='center', ha=ha, fontsize=9)

# ── Right: scatter of errors vs each feature to show alignment ─────────────────
corrs = [np.corrcoef(X5_sc[:,j], errors50)[0,1] for j in range(5)]
colors_c = ['#4CAF50' if c > 0 else '#e57373' for c in corrs]
bars2 = axes[1].barh(FEATURES2, corrs, color=colors_c, edgecolor='white', height=0.55)
axes[1].axvline(0, color='black', lw=1.2)
axes[1].set_xlabel('Pearson correlation  (errors vs feature column)')
axes[1].set_title('Correlated errors = learnable signal still remaining', fontsize=10)
for bar, val in zip(bars2, corrs):
    xpos = val + 0.01 if val >= 0 else val - 0.01
    ha   = 'left' if val >= 0 else 'right'
    axes[1].text(xpos, bar.get_y()+bar.get_height()/2,
                 f'{val:.3f}', va='center', ha=ha, fontsize=9)

plt.suptitle('Gradient = dot product of error vector with each feature column  —  measures remaining learnable signal',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print('Large dot product (errors aligned with feature) → large gradient → weight needs updating.')
print('Near zero dot product → errors have no pattern along that feature → weight is calibrated.')
print('Convergence = all dot products near zero = error vector perpendicular to all feature columns.')


## Key Takeaways

### The dot product is a single operation behind all of ML

```
a · b  =  Σ aⱼbⱼ  =  |a| × |b| × cos(θ)
```

| θ | cos(θ) | dot product | meaning |
|---|---|---|---|
| 0° | 1 | maximum positive | vectors fully aligned |
| 90° | 0 | zero | vectors perpendicular — no shared direction |
| 180° | -1 | maximum negative | vectors pointing opposite |

### Where the dot product appears in ML

| Concept | The dot product | What it measures |
|---|---|---|
| Linear regression prediction | `w · x` | How aligned is this observation with the learned pattern? |
| Neural network neuron | `w · x + b → activation` | Does input activate this pattern detector? |
| Transformer attention | `q · k / √d` | How relevant is this key token to the query? |
| Word embeddings | cosine similarity = `â · b̂` | Do these concepts point in the same direction? |
| Gradient update | `errors · column_j` | Are errors still correlated with this feature? |

### The silent feature rule is geometry

A zero feature value means the observation has no component in that dimension.
`w_j × 0 = 0` is not a special rule — it is the dot product of a vector with a zero-component axis.

### Convergence is perpendicularity

Training ends when the error vector is perpendicular to every feature column:

```
errors · column_j  ≈  0   for all j
```

At this point, no more dot product signal remains — the weights have absorbed all the patterns the data contains.

### The unifying insight

The arithmetic that makes a prediction (`w · x`) and the arithmetic that improves weights (`errors · x_j`) are the same dot product. Machine learning is geometry — finding the weight vector that best aligns with the patterns in data.
